# LCDB v6.0 Land-Cover Context Experiment

This notebook tests whether land-cover context adds predictive information
for classifying the severity of recorded New Zealand road crashes.

It uses the 2023 land-cover attributes from LCDB v6.0 and compares
land-cover feature configurations with the established CAS baseline.

The target is `crashSeverity`, with four classes: Non-Injury, Minor,
Serious and Fatal.

The analysis classifies recorded crashes; it does not predict crash
occurrence or future crash risk.

In [1]:
# Inspect LCDB v6.0 GeoPackage

import geopandas as gpd
from pathlib import Path


# 1. Find project root automatically

CURRENT_FOLDER = Path.cwd()

PROJECT_ROOT = None

for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]:

    candidate = (
        folder
        / "data"
        / "lcdb"
        / "LCDB_v6.gpkg"
    )

    if candidate.exists():
        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "data/lcdb/LCDB_v6.gpkg."
    )


LCDB_PATH = (
    PROJECT_ROOT
    / "data"
    / "lcdb"
    / "LCDB_v6.gpkg"
)


# 2. Confirm file

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "File exists:",
    LCDB_PATH.exists()
)

print(
    "\nFile path:"
)

print(
    LCDB_PATH
)


if not LCDB_PATH.exists():
    raise FileNotFoundError(
        f"LCDB GeoPackage not found:\n{LCDB_PATH}"
    )


print(
    "\nFile size:",
    round(
        LCDB_PATH.stat().st_size
        / 1024
        / 1024
        / 1024,
        2
    ),
    "GB"
)


# 3. Inspect layers without loading the entire file

layers = gpd.list_layers(
    LCDB_PATH
)

print(
    "\nGeoPackage Layers"
)

display(
    layers
)


# 4. Read only the first 5 records from each layer

for layer_name in layers["name"]:

    print(
        "\nLayer:",
        layer_name
    )

    sample = gpd.read_file(
        LCDB_PATH,
        layer=layer_name,
        rows=5
    )

    print(
        "\nCRS:"
    )

    print(
        sample.crs
    )

    print(
        "\nColumns:"
    )

    print(
        sample.columns.tolist()
    )

    print(
        "\nGeometry types in sample:"
    )

    print(
        sample.geometry
        .geom_type
        .value_counts(
            dropna=False
        )
    )

    print(
        "\nFirst rows:"
    )

    display(
        sample.drop(
            columns="geometry",
            errors="ignore"
        )
    )

Project root: C:\Users\ASUS\Desktop\Project_Roadcrash
File exists: True

File path:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\lcdb\LCDB_v6.gpkg

File size: 1.31 GB

GeoPackage Layers


,name,geometry_type
0,lcdb_v60_land_cover_database_version_60_mainla...,MultiPolygon



Layer: lcdb_v60_land_cover_database_version_60_mainland_new_zealand

CRS:
EPSG:2193

Columns:
['Name_2023', 'Name_2018', 'Name_2012', 'Name_2008', 'Name_2001', 'Name_1996', 'Class_2023', 'Class_2018', 'Class_2012', 'Class_2008', 'Class_2001', 'Class_1996', 'Wetland_23', 'Wetland_18', 'Wetland_12', 'Wetland_08', 'Wetland_01', 'Wetland_96', 'Onshore_23', 'Onshore_18', 'Onshore_12', 'Onshore_08', 'Onshore_01', 'Onshore_96', 'EditAuthor', 'EditDate', 'LCDB_UID', 'geometry']

Geometry types in sample:
MultiPolygon    5
Name: count, dtype: int64

First rows:


,Name_2023,Name_2018,Name_2012,Name_2008,Name_2001,Name_1996,Class_2023,Class_2018,Class_2012,Class_2008,...,Wetland_96,Onshore_23,Onshore_18,Onshore_12,Onshore_08,Onshore_01,Onshore_96,EditAuthor,EditDate,LCDB_UID
0,Not land,Sand and Gravel,Sand and Gravel,Sand and Gravel,Sand and Gravel,Sand and Gravel,0,10,10,10,...,no,no,yes,yes,yes,yes,yes,Landcare Research,2024-08-01,lcdb2000600005
1,Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),1,1,1,1,...,no,yes,yes,yes,yes,yes,yes,Terralink,2004-06-30,lcdb2000000012
2,Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),1,1,1,1,...,no,yes,yes,yes,yes,yes,yes,MfE (LUM),2013-07-01,lcdb2000001466
3,Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),1,1,1,1,...,no,yes,yes,yes,yes,yes,yes,Landcare Research,2004-06-30,lcdb2000000021
4,Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),1,1,1,1,...,no,yes,yes,yes,yes,yes,yes,Landcare Research,2004-06-30,lcdb2000000050


In [2]:
# Inspect LCDB v6.0 2023 land-cover attributes
# without loading full polygon geometry

import pandas as pd
import pyogrio


LCDB_LAYER = (
    "lcdb_v60_land_cover_database_version_60_mainland_new_zealand"
)


# 1. Read layer metadata

info = pyogrio.read_info(
    LCDB_PATH,
    layer=LCDB_LAYER
)

print(
    "LCDB Layer Information"
)

print(
    "Feature count:",
    info["features"]
)

print(
    "CRS:",
    info["crs"]
)

print(
    "Geometry type:",
    info["geometry_type"]
)


# 2. Read only 2023 attributes
# No polygon geometries yet

lcdb_attributes = pyogrio.read_dataframe(
    LCDB_PATH,
    layer=LCDB_LAYER,
    columns=[
        "Name_2023",
        "Class_2023",
        "Wetland_23",
        "Onshore_23",
        "LCDB_UID"
    ],
    read_geometry=False
)


print(
    "\nAttribute table shape:",
    lcdb_attributes.shape
)


# 3. Missingness

print(
    "\nMissing Values"
)

print(
    lcdb_attributes[
        [
            "Name_2023",
            "Class_2023",
            "Wetland_23",
            "Onshore_23"
        ]
    ]
    .isna()
    .sum()
)


# 4. Unique 2023 land-cover classes

print(
    "\nUnique Class_2023 codes:",
    lcdb_attributes[
        "Class_2023"
    ]
    .nunique(
        dropna=True
    )
)

print(
    "Unique Name_2023 classes:",
    lcdb_attributes[
        "Name_2023"
    ]
    .nunique(
        dropna=True
    )
)


# 5. Full class distribution

class_distribution = (
    lcdb_attributes
    .groupby(
        [
            "Class_2023",
            "Name_2023"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="Polygon_Count"
    )
    .sort_values(
        "Class_2023"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nLCDB 2023 Land-Cover Classes"
)

display(
    class_distribution
)


# 6. Onshore distribution

print(
    "\nOnshore_23 Distribution"
)

print(
    lcdb_attributes[
        "Onshore_23"
    ]
    .value_counts(
        dropna=False
    )
)


# 7. Wetland distribution

print(
    "\nWetland_23 Distribution"
)

print(
    lcdb_attributes[
        "Wetland_23"
    ]
    .value_counts(
        dropna=False
    )
)

LCDB Layer Information
Feature count: 542789
CRS: EPSG:2193
Geometry type: MultiPolygon

Attribute table shape: (542789, 5)

Missing Values
Name_2023     0
Class_2023    0
Wetland_23    0
Onshore_23    0
dtype: int64

Unique Class_2023 codes: 34
Unique Name_2023 classes: 34

LCDB 2023 Land-Cover Classes


,Class_2023,Name_2023,Polygon_Count
0,0,Not land,2
1,1,Built-up Area (settlement),10171
2,2,Urban Parkland/Open Space,4608
3,5,Transport Infrastructure,1190
4,6,Surface Mine or Dump,2150
5,10,Sand and Gravel,3702
6,12,Landslide,7384
7,14,Permanent Snow and Ice,2283
8,15,Alpine Grass/Herbfield,5897
9,16,Gravel and Rock,30751



Onshore_23 Distribution
Onshore_23
yes    539976
no       2813
Name: count, dtype: int64

Wetland_23 Distribution
Wetland_23
no     519059
yes     23730
Name: count, dtype: int64


In [3]:
# Create broader LCDB environmental land-cover groups

import pandas as pd


# 1. Map the 34 LCDB classes into broader environmental-context groups

lcdb_group_map = {

    # Built / developed environment
    1: "Built / Urban",
    2: "Built / Urban",
    5: "Built / Urban",

    # Agriculture / horticulture
    30: "Cropland / Horticulture",
    33: "Cropland / Horticulture",

    # Grassland
    15: "Grassland",
    40: "Grassland",
    41: "Grassland",
    43: "Grassland",
    44: "Grassland",

    # Shrubland / scrub / woody vegetation
    47: "Shrubland / Scrub",
    50: "Shrubland / Scrub",
    51: "Shrubland / Scrub",
    52: "Shrubland / Scrub",
    54: "Shrubland / Scrub",
    55: "Shrubland / Scrub",
    56: "Shrubland / Scrub",
    58: "Shrubland / Scrub",

    # Forest
    64: "Forest",
    68: "Forest",
    69: "Forest",
    71: "Forest",

    # Water / wetland environment
    20: "Water / Wetland",
    21: "Water / Wetland",
    22: "Water / Wetland",
    45: "Water / Wetland",
    46: "Water / Wetland",
    70: "Water / Wetland",

    # Bare / disturbed / alpine terrain
    6: "Bare / Disturbed",
    10: "Bare / Disturbed",
    12: "Bare / Disturbed",
    14: "Bare / Disturbed",
    16: "Bare / Disturbed",

    # Not land
    0: "Other / Not Land"
}


# 2. Create grouped variable

lcdb_attributes[
    "LCDB_Group"
] = (
    lcdb_attributes[
        "Class_2023"
    ]
    .map(
        lcdb_group_map
    )
)


# 3. Validate that every LCDB class was mapped

unmapped = (
    lcdb_attributes.loc[
        lcdb_attributes[
            "LCDB_Group"
        ].isna(),
        [
            "Class_2023",
            "Name_2023"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Class_2023"
    )
)


print(
    "Unmapped LCDB Classes"
)

display(
    unmapped
)


# 4. Detailed class-to-broad-group lookup

class_group_lookup = (
    lcdb_attributes[
        [
            "Class_2023",
            "Name_2023",
            "LCDB_Group"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Class_2023"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nLCDB Class Grouping"
)

display(
    class_group_lookup
)


# 5. Polygon distribution by broad group

group_distribution = (
    lcdb_attributes[
        "LCDB_Group"
    ]
    .value_counts()
    .rename_axis(
        "LCDB_Group"
    )
    .reset_index(
        name="Polygon_Count"
    )
)


group_distribution[
    "Polygon_Percent"
] = (
    group_distribution[
        "Polygon_Count"
    ]
    / len(lcdb_attributes)
    * 100
).round(2)


print(
    "\nBroad LCDB Group Distribution"
)

display(
    group_distribution
)


# 6. Final validation

print(
    "\nTotal polygons:",
    len(lcdb_attributes)
)

print(
    "Mapped polygons:",
    lcdb_attributes[
        "LCDB_Group"
    ]
    .notna()
    .sum()
)

print(
    "Unmapped polygons:",
    lcdb_attributes[
        "LCDB_Group"
    ]
    .isna()
    .sum()
)

print(
    "Broad land-cover groups:",
    lcdb_attributes[
        "LCDB_Group"
    ]
    .nunique()
)

Unmapped LCDB Classes


,Class_2023,Name_2023



LCDB Class Grouping


,Class_2023,Name_2023,LCDB_Group
0,0,Not land,Other / Not Land
1,1,Built-up Area (settlement),Built / Urban
2,2,Urban Parkland/Open Space,Built / Urban
3,5,Transport Infrastructure,Built / Urban
4,6,Surface Mine or Dump,Bare / Disturbed
5,10,Sand and Gravel,Bare / Disturbed
6,12,Landslide,Bare / Disturbed
7,14,Permanent Snow and Ice,Bare / Disturbed
8,15,Alpine Grass/Herbfield,Grassland
9,16,Gravel and Rock,Bare / Disturbed



Broad LCDB Group Distribution


,LCDB_Group,Polygon_Count,Polygon_Percent
0,Forest,200830,37.00
1,Shrubland / Scrub,149094,27.47
2,Grassland,86942,16.02
3,Bare / Disturbed,46270,8.52
4,Water / Wetland,32902,6.06
5,Built / Urban,15969,2.94
6,Cropland / Horticulture,10780,1.99
7,Other / Not Land,2,0.00



Total polygons: 542789
Mapped polygons: 542789
Unmapped polygons: 0
Broad land-cover groups: 8


In [4]:
# Spatially assign LCDB v6.0 land cover to CAS crashes
# Memory-safe spatial chunk processing

import gc

import geopandas as gpd
import pandas as pd
import numpy as np
import pyogrio


# 1. Load original CAS dataset first

CAS_PATH = (
    PROJECT_ROOT
    / "data"
    / "CAS_Data_public.csv"
)


cas = pd.read_csv(
    CAS_PATH,
    low_memory=False
)


cas[
    "cas_row_id"
] = np.arange(
    len(cas)
)


print(
    "CAS rows:",
    len(cas)
)


# 2. Create crash points
# X/Y are NZTM2000 coordinates, EPSG:2193

valid_xy = (
    pd.to_numeric(
        cas["X"],
        errors="coerce"
    ).notna()
    &
    pd.to_numeric(
        cas["Y"],
        errors="coerce"
    ).notna()
)


cas_valid = (
    cas.loc[
        valid_xy
    ]
    .copy()
)


cas_invalid = (
    cas.loc[
        ~valid_xy
    ]
    .copy()
)


crash_points = gpd.GeoDataFrame(
    cas_valid[
        [
            "cas_row_id",
            "X",
            "Y",
            "crashSeverity"
        ]
    ].copy(),

    geometry=gpd.points_from_xy(
        cas_valid["X"],
        cas_valid["Y"]
    ),

    crs="EPSG:2193"
)


print(
    "\nValid crash points:",
    len(crash_points)
)

print(
    "Crashes with missing coordinates:",
    len(cas_invalid)
)

print(
    "Crash CRS:",
    crash_points.crs
)


# 3. Confirm LCDB layer CRS

lcdb_info = pyogrio.read_info(
    LCDB_PATH,
    layer=LCDB_LAYER
)


print(
    "\nLCDB CRS:",
    lcdb_info["crs"]
)


if "2193" not in str(
    lcdb_info["crs"]
):

    raise ValueError(
        "LCDB layer is not in EPSG:2193. "
        "The crash coordinates and LCDB layer must use "
        "the same CRS before chunk processing."
    )


# 4. Create spatial grid identifiers
# LCDB will be read only around grid cells containing crashes

GRID_SIZE_M = 20_000
BBOX_BUFFER_M = 1_000


crash_points[
    "_grid_x"
] = np.floor(
    crash_points.geometry.x
    / GRID_SIZE_M
).astype(int)


crash_points[
    "_grid_y"
] = np.floor(
    crash_points.geometry.y
    / GRID_SIZE_M
).astype(int)


grid_groups = list(
    crash_points.groupby(
        [
            "_grid_x",
            "_grid_y"
        ],
        sort=True
    )
)


print(
    "\nOccupied crash grid cells:",
    len(grid_groups)
)

print(
    "Grid size:",
    GRID_SIZE_M,
    "m"
)


# 5. Process one crash grid at a time

joined_parts = []
lcdb_parts = []


print(
    "\nSpatially matching crashes to LCDB..."
)


for i, (
    (grid_x, grid_y),
    crash_chunk
) in enumerate(
    grid_groups,
    start=1
):

    west = (
        grid_x
        * GRID_SIZE_M
        - BBOX_BUFFER_M
    )

    south = (
        grid_y
        * GRID_SIZE_M
        - BBOX_BUFFER_M
    )

    east = (
        (grid_x + 1)
        * GRID_SIZE_M
        + BBOX_BUFFER_M
    )

    north = (
        (grid_y + 1)
        * GRID_SIZE_M
        + BBOX_BUFFER_M
    )


    local_lcdb = pyogrio.read_dataframe(
        LCDB_PATH,
        layer=LCDB_LAYER,
        columns=[
            "Name_2023",
            "Class_2023",
            "Wetland_23",
            "Onshore_23",
            "LCDB_UID"
        ],
        bbox=(
            west,
            south,
            east,
            north
        )
    )


    if len(
        local_lcdb
    ) > 0:

        local_lcdb[
            "LCDB_Group"
        ] = (
            local_lcdb[
                "Class_2023"
            ]
            .map(
                lcdb_group_map
            )
        )


        local_join = gpd.sjoin(
            crash_chunk[
                [
                    "cas_row_id",
                    "crashSeverity",
                    "geometry"
                ]
            ],

            local_lcdb[
                [
                    "Name_2023",
                    "Class_2023",
                    "LCDB_Group",
                    "Wetland_23",
                    "Onshore_23",
                    "LCDB_UID",
                    "geometry"
                ]
            ],

            how="left",
            predicate="within"
        )


        joined_parts.append(
            local_join
        )


        lcdb_parts.append(
            local_lcdb[
                [
                    "Name_2023",
                    "Class_2023",
                    "LCDB_Group",
                    "Wetland_23",
                    "Onshore_23",
                    "LCDB_UID",
                    "geometry"
                ]
            ].copy()
        )


    else:

        local_join = crash_chunk[
            [
                "cas_row_id",
                "crashSeverity",
                "geometry"
            ]
        ].copy()


        local_join[
            "Name_2023"
        ] = np.nan

        local_join[
            "Class_2023"
        ] = np.nan

        local_join[
            "LCDB_Group"
        ] = np.nan

        local_join[
            "Wetland_23"
        ] = np.nan

        local_join[
            "Onshore_23"
        ] = np.nan

        local_join[
            "LCDB_UID"
        ] = np.nan

        local_join[
            "index_right"
        ] = np.nan


        joined_parts.append(
            local_join
        )


    del local_lcdb
    del local_join


    if (
        i % 25 == 0
        or i == len(grid_groups)
    ):

        print(
            f"Processed grid cells: "
            f"{i}/{len(grid_groups)}"
        )


    if i % 25 == 0:
        gc.collect()


# 6. Combine crash-level spatial matches

crash_lcdb = gpd.GeoDataFrame(
    pd.concat(
        joined_parts,
        ignore_index=True
    ),
    geometry="geometry",
    crs="EPSG:2193"
)


crash_lcdb = (
    crash_lcdb
    .sort_values(
        "cas_row_id"
    )
    .reset_index(
        drop=True
    )
)


# 7. Build reduced LCDB spatial dataset
# This keeps only polygons from areas around crash locations

if lcdb_parts:

    lcdb_spatial = gpd.GeoDataFrame(
        pd.concat(
            lcdb_parts,
            ignore_index=True
        ),
        geometry="geometry",
        crs="EPSG:2193"
    )


    lcdb_spatial = (
        lcdb_spatial
        .drop_duplicates(
            subset=[
                "LCDB_UID"
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    lcdb_spatial = gpd.GeoDataFrame(
        columns=[
            "Name_2023",
            "Class_2023",
            "LCDB_Group",
            "Wetland_23",
            "Onshore_23",
            "LCDB_UID",
            "geometry"
        ],
        geometry="geometry",
        crs="EPSG:2193"
    )


print(
    "\nReduced LCDB polygons retained:",
    len(lcdb_spatial)
)

print(
    "LCDB CRS:",
    lcdb_spatial.crs
)

print(
    "Missing LCDB geometry:",
    lcdb_spatial.geometry
    .isna()
    .sum()
)


# 8. Add crashes that have missing coordinates

if len(
    cas_invalid
) > 0:

    invalid_rows = gpd.GeoDataFrame(
        {
            "cas_row_id":
                cas_invalid[
                    "cas_row_id"
                ].to_numpy(),

            "crashSeverity":
                cas_invalid[
                    "crashSeverity"
                ].to_numpy(),

            "Name_2023":
                np.nan,

            "Class_2023":
                np.nan,

            "LCDB_Group":
                np.nan,

            "Wetland_23":
                np.nan,

            "Onshore_23":
                np.nan,

            "LCDB_UID":
                np.nan,

            "index_right":
                np.nan
        },

        geometry=[
            None
        ] * len(
            cas_invalid
        ),

        crs="EPSG:2193"
    )


    crash_lcdb = gpd.GeoDataFrame(
        pd.concat(
            [
                crash_lcdb,
                invalid_rows
            ],
            ignore_index=True
        ),
        geometry="geometry",
        crs="EPSG:2193"
    )


    crash_lcdb = (
        crash_lcdb
        .sort_values(
            "cas_row_id"
        )
        .reset_index(
            drop=True
        )
    )


# 9. Check unmapped LCDB polygons

print(
    "\nUnmapped LCDB polygons:",
    lcdb_spatial[
        "LCDB_Group"
    ]
    .isna()
    .sum()
)


# 10. Check duplicate polygon matches

match_counts = (
    crash_lcdb[
        "cas_row_id"
    ]
    .value_counts()
)


multiple_matches = (
    match_counts > 1
).sum()


print(
    "\nCrash points matched to multiple LCDB polygons:",
    multiple_matches
)


# 11. Coverage statistics

matched = (
    crash_lcdb[
        "Name_2023"
    ]
    .notna()
)


matched_crash_ids = (
    crash_lcdb.loc[
        matched,
        "cas_row_id"
    ]
    .nunique()
)


total_crashes = (
    cas[
        "cas_row_id"
    ]
    .nunique()
)


unmatched_crashes = (
    total_crashes
    - matched_crash_ids
)


print(
    "\nLCDB Spatial Match Results"
)

print(
    "Original crash points:",
    total_crashes
)

print(
    "Joined rows:",
    len(crash_lcdb)
)

print(
    "Matched crashes:",
    matched_crash_ids
)

print(
    "Unmatched crashes:",
    unmatched_crashes
)

print(
    "Match rate:",
    round(
        matched_crash_ids
        / total_crashes
        * 100,
        2
    ),
    "%"
)


# 12. Crash-level detailed land-cover distribution

print(
    "\nCrash-Level Detailed LCDB Distribution"
)


detailed_crash_distribution = (
    crash_lcdb[
        "Name_2023"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    detailed_crash_distribution
)


# 13. Crash-level broad land-cover distribution

print(
    "\nCrash-Level Broad LCDB Distribution"
)


broad_crash_distribution = (
    crash_lcdb[
        "LCDB_Group"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    broad_crash_distribution
)


# 14. Wetland distribution among crashes

print(
    "\nCrash-Level Wetland Distribution"
)


print(
    crash_lcdb[
        "Wetland_23"
    ]
    .value_counts(
        dropna=False
    )
)


# 15. Onshore distribution among crashes

print(
    "\nCrash-Level Onshore Distribution"
)


print(
    crash_lcdb[
        "Onshore_23"
    ]
    .value_counts(
        dropna=False
    )
)


# 16. Clean temporary objects

del joined_parts
del lcdb_parts
del grid_groups

gc.collect()

CAS rows: 34866

Valid crash points: 34866
Crashes with missing coordinates: 0
Crash CRS: EPSG:2193

LCDB CRS: EPSG:2193

Occupied crash grid cells: 590
Grid size: 20000 m

Spatially matching crashes to LCDB...
Processed grid cells: 25/590
Processed grid cells: 50/590
Processed grid cells: 75/590
Processed grid cells: 100/590
Processed grid cells: 125/590
Processed grid cells: 150/590
Processed grid cells: 175/590
Processed grid cells: 200/590
Processed grid cells: 225/590
Processed grid cells: 250/590
Processed grid cells: 275/590
Processed grid cells: 300/590
Processed grid cells: 325/590
Processed grid cells: 350/590
Processed grid cells: 375/590
Processed grid cells: 400/590
Processed grid cells: 425/590
Processed grid cells: 450/590
Processed grid cells: 475/590
Processed grid cells: 500/590
Processed grid cells: 525/590
Processed grid cells: 550/590
Processed grid cells: 575/590
Processed grid cells: 590/590

Reduced LCDB polygons retained: 451224
LCDB CRS: EPSG:2193
Missing LCDB

0

In [5]:
# Investigate unmatched LCDB crash points
# using nearest LCDB polygon distance

import geopandas as gpd
import pandas as pd


# 1. Extract unmatched crash points

unmatched_ids = crash_lcdb.loc[
    crash_lcdb[
        "Name_2023"
    ].isna(),
    "cas_row_id"
].unique()


unmatched_points = crash_points[
    crash_points[
        "cas_row_id"
    ]
    .isin(
        unmatched_ids
    )
].copy()


print(
    "Unmatched crash points:",
    len(unmatched_points)
)


# 2. Find nearest LCDB polygon

nearest_lcdb = gpd.sjoin_nearest(
    unmatched_points,
    lcdb_spatial[
        [
            "Name_2023",
            "Class_2023",
            "LCDB_Group",
            "Wetland_23",
            "Onshore_23",
            "LCDB_UID",
            "geometry"
        ]
    ],
    how="left",
    distance_col="LCDB_distance_m"
)


# 3. Check duplicate nearest matches

nearest_counts = (
    nearest_lcdb[
        "cas_row_id"
    ]
    .value_counts()
)

print(
    "\nCrash points with multiple equally-nearest polygons:",
    (nearest_counts > 1).sum()
)


# 4. Distance summary

print(
    "\nNearest LCDB Distance Summary"
)

print(
    nearest_lcdb[
        "LCDB_distance_m"
    ].describe()
)


# 5. Distance thresholds

for threshold in [
    1,
    2,
    5,
    10,
    25,
    50,
    100,
    250,
    500
]:

    count = (
        nearest_lcdb[
            "LCDB_distance_m"
        ]
        <= threshold
    ).sum()

    print(
        f"Within {threshold:>3} m: "
        f"{count} / {len(nearest_lcdb)}"
    )


# 6. Show the farthest unmatched cases

print(
    "\nFarthest Unmatched Crashes"
)

display(
    nearest_lcdb[
        [
            "cas_row_id",
            "crashSeverity",
            "LCDB_distance_m",
            "Name_2023",
            "LCDB_Group"
        ]
    ]
    .sort_values(
        "LCDB_distance_m",
        ascending=False
    )
    .head(20)
)

Unmatched crash points: 63

Crash points with multiple equally-nearest polygons: 0

Nearest LCDB Distance Summary
count        63.000000
mean      43701.292465
std      168619.585963
min           2.419263
25%          24.159290
50%          85.047478
75%         331.573177
max      691528.877750
Name: LCDB_distance_m, dtype: float64
Within   1 m: 0 / 63
Within   2 m: 0 / 63
Within   5 m: 3 / 63
Within  10 m: 7 / 63
Within  25 m: 17 / 63
Within  50 m: 24 / 63
Within 100 m: 33 / 63
Within 250 m: 44 / 63
Within 500 m: 59 / 63

Farthest Unmatched Crashes


,cas_row_id,crashSeverity,LCDB_distance_m,Name_2023,LCDB_Group
10078,10078,Serious Crash,691528.877750,Sand and Gravel,Bare / Disturbed
18308,18308,Minor Crash,689939.037161,Sand and Gravel,Bare / Disturbed
25135,25135,Non-Injury Crash,683450.910128,Sand and Gravel,Bare / Disturbed
21608,21608,Non-Injury Crash,679559.222148,Sand and Gravel,Bare / Disturbed
9054,9054,Non-Injury Crash,464.014863,Transport Infrastructure,Built / Urban
28552,28552,Non-Injury Crash,435.055551,Transport Infrastructure,Built / Urban
16831,16831,Non-Injury Crash,430.066986,Transport Infrastructure,Built / Urban
5455,5455,Non-Injury Crash,429.944318,Transport Infrastructure,Built / Urban
18366,18366,Non-Injury Crash,420.855821,Transport Infrastructure,Built / Urban
17446,17446,Non-Injury Crash,413.008202,Transport Infrastructure,Built / Urban


In [6]:
# Investigate unmatched LCDB crashes using CAS
# location information

# Select useful CAS context fields if they exist

context_candidates = [
    "cas_row_id",
    "X",
    "Y",
    "crashSeverity",
    "region",
    "tlaName",
    "crashLocation1",
    "crashLocation2",
    "crashSHDescription",
    "urban"
]

context_cols = [
    col
    for col in context_candidates
    if col in cas.columns
]


# Merge CAS location information with nearest results

unmatched_diagnostic = (
    nearest_lcdb[
        [
            "cas_row_id",
            "LCDB_distance_m",
            "Name_2023",
            "LCDB_Group"
        ]
    ]
    .merge(
        cas[
            context_cols
        ],
        on="cas_row_id",
        how="left"
    )
)


print(
    "Available Diagnostic Columns"
)

print(
    unmatched_diagnostic.columns.tolist()
)


# 1. Very distant cases

print(
    "\nUnmatched Crashes More Than 500 m "
    "from LCDB"
)

display(
    unmatched_diagnostic.loc[
        unmatched_diagnostic[
            "LCDB_distance_m"
        ] > 500
    ]
    .sort_values(
        "LCDB_distance_m",
        ascending=False
    )
)


# 2. Cases between 100 m and 500 m

print(
    "\nUnmatched Crashes 100–500 m "
    "from LCDB"
)

display(
    unmatched_diagnostic.loc[
        unmatched_diagnostic[
            "LCDB_distance_m"
        ].between(
            100,
            500,
            inclusive="right"
        )
    ]
    .sort_values(
        "LCDB_distance_m",
        ascending=False
    )
)


# 3. Distance groups

distance_bins = [
    -1,
    10,
    25,
    50,
    100,
    250,
    500,
    float("inf")
]

distance_labels = [
    "<=10 m",
    "10–25 m",
    "25–50 m",
    "50–100 m",
    "100–250 m",
    "250–500 m",
    ">500 m"
]

unmatched_diagnostic[
    "distance_group"
] = pd.cut(
    unmatched_diagnostic[
        "LCDB_distance_m"
    ],
    bins=distance_bins,
    labels=distance_labels
)


print(
    "\nDistance Group Counts"
)

print(
    unmatched_diagnostic[
        "distance_group"
    ]
    .value_counts()
    .sort_index()
)

Available Diagnostic Columns
['cas_row_id', 'LCDB_distance_m', 'Name_2023', 'LCDB_Group', 'X', 'Y', 'crashSeverity', 'region', 'tlaName', 'crashLocation1', 'crashLocation2', 'crashSHDescription', 'urban']

Unmatched Crashes More Than 500 m from LCDB


,cas_row_id,LCDB_distance_m,Name_2023,LCDB_Group,X,Y,crashSeverity,region,tlaName,crashLocation1,crashLocation2,crashSHDescription,urban
20,10078,691528.877750,Sand and Gravel,Bare / Disturbed,2.444711e+06,5.076666e+06,Serious Crash,Manawatū-Whanganui Region,Chatham Islands Territory,WAITANGI WHARF OWENGA ROAD,TE MATARAE ROAD,No,Open
37,18308,689939.037161,Sand and Gravel,Bare / Disturbed,2.443069e+06,5.077164e+06,Minor Crash,Manawatū-Whanganui Region,Chatham Islands Territory,WAITANGI WHARF OWENGA ROAD,TE MATARAE ROAD,No,Open
46,25135,683450.910128,Sand and Gravel,Bare / Disturbed,2.437698e+06,5.080842e+06,Non-Injury Crash,Manawatū-Whanganui Region,Chatham Islands Territory,TIKITIKI HILL ROAD,WAITANGI WHARF OWENGA ROAD,No,Urban
43,21608,679559.222148,Sand and Gravel,Bare / Disturbed,2.441217e+06,5.091483e+06,Non-Injury Crash,Manawatū-Whanganui Region,Chatham Islands Territory,NORTH ROAD,AIR BASE ROAD,No,Open



Unmatched Crashes 100–500 m from LCDB


,cas_row_id,LCDB_distance_m,Name_2023,LCDB_Group,X,Y,crashSeverity,region,tlaName,crashLocation1,crashLocation2,crashSHDescription,urban
19,9054,464.014863,Transport Infrastructure,Built / Urban,1.755633e+06,5.922446e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,CURRAN STREET ON RAMP,Yes,Open
52,28552,435.055551,Transport Infrastructure,Built / Urban,1.755643e+06,5.922416e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
34,16831,430.066986,Transport Infrastructure,Built / Urban,1.755670e+06,5.922465e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
11,5455,429.944318,Transport Infrastructure,Built / Urban,1.755670e+06,5.922465e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
38,18366,420.855821,Transport Infrastructure,Built / Urban,1.755610e+06,5.922406e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
35,17446,413.008202,Transport Infrastructure,Built / Urban,1.755606e+06,5.922398e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
36,18306,408.573528,Transport Infrastructure,Built / Urban,1.755601e+06,5.922395e+06,Minor Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
25,12330,402.751262,Transport Infrastructure,Built / Urban,1.755663e+06,5.922500e+06,Minor Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
23,11375,398.720037,Transport Infrastructure,Built / Urban,1.755606e+06,5.922384e+06,Non-Injury Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open
32,15136,380.746390,Transport Infrastructure,Built / Urban,1.755687e+06,5.922512e+06,Minor Crash,NaN,Auckland,NORTHERN MOTORWAY,TOP OF HARBOUR BRIDGE,Yes,Open



Distance Group Counts
distance_group
<=10 m        7
10–25 m      10
25–50 m       7
50–100 m      9
100–250 m    11
250–500 m    15
>500 m        4
Name: count, dtype: int64


In [7]:
# Finalise crash-level LCDB features
# Keep exact spatial matches only

import numpy as np
import pandas as pd


# 1. Create clean crash-level LCDB dataset

lcdb_crash_final = (
    crash_lcdb[
        [
            "cas_row_id",
            "crashSeverity",
            "Name_2023",
            "Class_2023",
            "LCDB_Group",
            "Wetland_23",
            "Onshore_23",
            "LCDB_UID"
        ]
    ]
    .copy()
)


# 2. Explicit category for crashes with no exact
# LCDB polygon match

UNMATCHED_LABEL = (
    "No LCDB polygon / outside coverage"
)


lcdb_crash_final[
    "LCDB_Name_clean"
] = (
    lcdb_crash_final[
        "Name_2023"
    ]
    .fillna(
        UNMATCHED_LABEL
    )
)


lcdb_crash_final[
    "LCDB_Group_clean"
] = (
    lcdb_crash_final[
        "LCDB_Group"
    ]
    .fillna(
        UNMATCHED_LABEL
    )
)


# 3. Spatial-match indicator

lcdb_crash_final[
    "LCDB_exact_match"
] = (
    lcdb_crash_final[
        "Name_2023"
    ]
    .notna()
    .astype(int)
)


# 4. Validation

print(
    "Final LCDB Crash Dataset"
)

print(
    "Rows:",
    len(lcdb_crash_final)
)

print(
    "Unique crash IDs:",
    lcdb_crash_final[
        "cas_row_id"
    ].nunique()
)

print(
    "Duplicate crash IDs:",
    lcdb_crash_final[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


print(
    "\nExact LCDB matches:"
)

print(
    lcdb_crash_final[
        "LCDB_exact_match"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nMissing clean detailed class:",
    lcdb_crash_final[
        "LCDB_Name_clean"
    ]
    .isna()
    .sum()
)

print(
    "Missing clean broad group:",
    lcdb_crash_final[
        "LCDB_Group_clean"
    ]
    .isna()
    .sum()
)


# 5. Final broad group crash distribution

final_group_distribution = (
    lcdb_crash_final[
        "LCDB_Group_clean"
    ]
    .value_counts()
    .rename_axis(
        "LCDB_Group"
    )
    .reset_index(
        name="Crash_Count"
    )
)


final_group_distribution[
    "Crash_Percent"
] = (
    final_group_distribution[
        "Crash_Count"
    ]
    / len(lcdb_crash_final)
    * 100
).round(2)


print(
    "\nFinal LCDB Crash Distribution"
)

display(
    final_group_distribution
)


# 6. Severity × broad land-cover group

severity_group_counts = pd.crosstab(
    lcdb_crash_final[
        "LCDB_Group_clean"
    ],
    lcdb_crash_final[
        "crashSeverity"
    ]
)


severity_group_percent = (
    pd.crosstab(
        lcdb_crash_final[
            "LCDB_Group_clean"
        ],
        lcdb_crash_final[
            "crashSeverity"
        ],
        normalize="index"
    )
    * 100
)


print(
    "\nSeverity Counts by LCDB Group"
)

display(
    severity_group_counts
)


print(
    "\nSeverity Percentages Within LCDB Group"
)

display(
    severity_group_percent.round(2)
)

Final LCDB Crash Dataset
Rows: 34866
Unique crash IDs: 34866
Duplicate crash IDs: 0

Exact LCDB matches:
LCDB_exact_match
0       63
1    34803
Name: count, dtype: int64

Missing clean detailed class: 0
Missing clean broad group: 0

Final LCDB Crash Distribution


,LCDB_Group,Crash_Count,Crash_Percent
0,Built / Urban,23547,67.54
1,Grassland,8574,24.59
2,Forest,1007,2.89
3,Shrubland / Scrub,864,2.48
4,Cropland / Horticulture,539,1.55
5,Water / Wetland,172,0.49
6,Bare / Disturbed,100,0.29
7,No LCDB polygon / outside coverage,63,0.18



Severity Counts by LCDB Group


crashSeverity,Fatal Crash,Minor Crash,Non-Injury Crash,Serious Crash
LCDB_Group_clean,,,,
Bare / Disturbed,3,29,55,13
Built / Urban,100,6507,15647,1293
Cropland / Horticulture,9,168,304,58
Forest,12,289,582,124
Grassland,187,2699,4786,902
No LCDB polygon / outside coverage,1,17,30,15
Shrubland / Scrub,15,246,513,90
Water / Wetland,0,42,119,11



Severity Percentages Within LCDB Group


crashSeverity,Fatal Crash,Minor Crash,Non-Injury Crash,Serious Crash
LCDB_Group_clean,,,,
Bare / Disturbed,3.00,29.00,55.00,13.00
Built / Urban,0.42,27.63,66.45,5.49
Cropland / Horticulture,1.67,31.17,56.40,10.76
Forest,1.19,28.70,57.80,12.31
Grassland,2.18,31.48,55.82,10.52
No LCDB polygon / outside coverage,1.59,26.98,47.62,23.81
Shrubland / Scrub,1.74,28.47,59.38,10.42
Water / Wetland,0.00,24.42,69.19,6.40


In [8]:
# LCDB land-cover association with crash severity

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency


UNMATCHED_LABEL = (
    "No LCDB polygon / outside coverage"
)


# 1. Use only crashes with an exact LCDB land-cover match
# for environmental association analysis

lcdb_matched = (
    lcdb_crash_final.loc[
        lcdb_crash_final[
            "LCDB_Group_clean"
        ] != UNMATCHED_LABEL
    ]
    .copy()
)


print(
    "Matched crashes used for association analysis:",
    len(lcdb_matched)
)


# 2. Calculate Fatal + Serious percentage

lcdb_matched[
    "Severe_Outcome"
] = (
    lcdb_matched[
        "crashSeverity"
    ]
    .isin(
        [
            "Fatal Crash",
            "Serious Crash"
        ]
    )
)


severe_summary = (
    lcdb_matched
    .groupby(
        "LCDB_Group_clean"
    )
    .agg(
        Crash_Count=(
            "cas_row_id",
            "count"
        ),
        Severe_Count=(
            "Severe_Outcome",
            "sum"
        )
    )
)


severe_summary[
    "Severe_Percent"
] = (
    severe_summary[
        "Severe_Count"
    ]
    / severe_summary[
        "Crash_Count"
    ]
    * 100
).round(2)


severe_summary = (
    severe_summary
    .sort_values(
        "Severe_Percent",
        ascending=False
    )
)


print(
    "\nFatal + Serious Crash Percentage by LCDB Group"
)

display(
    severe_summary
)


# 3. Four-class severity × LCDB chi-square test

severity_table = pd.crosstab(
    lcdb_matched[
        "LCDB_Group_clean"
    ],
    lcdb_matched[
        "crashSeverity"
    ]
)


chi2_4, p_4, dof_4, expected_4 = (
    chi2_contingency(
        severity_table
    )
)


n_4 = (
    severity_table
    .to_numpy()
    .sum()
)

r_4, c_4 = (
    severity_table.shape
)


cramers_v_4 = np.sqrt(
    chi2_4
    /
    (
        n_4
        * min(
            r_4 - 1,
            c_4 - 1
        )
    )
)


print(
    "\nFour-Class Severity Association"
)

print(
    "Chi-square:",
    round(
        chi2_4,
        4
    )
)

print(
    "Degrees of freedom:",
    dof_4
)

print(
    "p-value:",
    p_4
)

print(
    "Cramer's V:",
    round(
        cramers_v_4,
        4
    )
)


expected_4_df = pd.DataFrame(
    expected_4,
    index=severity_table.index,
    columns=severity_table.columns
)


print(
    "\nExpected cells < 5:",
    (expected_4_df < 5)
    .sum()
    .sum()
)

print(
    "Minimum expected count:",
    round(
        expected_4_df.min().min(),
        4
    )
)


# 4. Additional severe vs non-severe association
# This is descriptive only.
# It does not change the four-class ML target.

binary_table = pd.crosstab(
    lcdb_matched[
        "LCDB_Group_clean"
    ],
    lcdb_matched[
        "Severe_Outcome"
    ]
)


chi2_binary, p_binary, dof_binary, expected_binary = (
    chi2_contingency(
        binary_table
    )
)


n_binary = (
    binary_table
    .to_numpy()
    .sum()
)

r_binary, c_binary = (
    binary_table.shape
)


cramers_v_binary = np.sqrt(
    chi2_binary
    /
    (
        n_binary
        * min(
            r_binary - 1,
            c_binary - 1
        )
    )
)


print(
    "\nSevere vs Non-Severe Association"
)

print(
    "Chi-square:",
    round(
        chi2_binary,
        4
    )
)

print(
    "Degrees of freedom:",
    dof_binary
)

print(
    "p-value:",
    p_binary
)

print(
    "Cramer's V:",
    round(
        cramers_v_binary,
        4
    )
)

print(
    "Expected cells < 5:",
    (expected_binary < 5)
    .sum()
)

print(
    "Minimum expected count:",
    round(
        expected_binary.min(),
        4
    )
)

Matched crashes used for association analysis: 34803

Fatal + Serious Crash Percentage by LCDB Group


,Crash_Count,Severe_Count,Severe_Percent
LCDB_Group_clean,,,
Bare / Disturbed,100,16,16.00
Forest,1007,136,13.51
Grassland,8574,1089,12.70
Cropland / Horticulture,539,67,12.43
Shrubland / Scrub,864,105,12.15
Water / Wetland,172,11,6.40
Built / Urban,23547,1393,5.92



Four-Class Severity Association
Chi-square: 675.6818
Degrees of freedom: 18
p-value: 8.168156594429215e-132
Cramer's V: 0.0804

Expected cells < 5: 2
Minimum expected count: 0.9367

Severe vs Non-Severe Association
Chi-square: 476.2954
Degrees of freedom: 6
p-value: 1.0717226595613205e-99
Cramer's V: 0.117
Expected cells < 5: 0
Minimum expected count: 8.0941


### LCDB Land-Cover Association with Crash Severity

Of the 34,866 recorded crashes, 34,803 (99.82%) were matched exactly to an
LCDB v6.0 polygon. The remaining 63 crashes were retained using an explicit
"No LCDB polygon / outside coverage" category rather than assigning potentially
incorrect nearest land-cover values.

Among crashes with an exact LCDB match, the proportion of Fatal or Serious
crashes varied substantially across broad land-cover environments. Built/Urban
locations had the lowest severe-crash proportion (5.92%), compared with
Grassland (12.70%), Forest (13.51%), Cropland/Horticulture (12.43%), and
Shrubland/Scrub (12.15%). Bare/Disturbed locations had a higher severe-crash
proportion (16.00%), although this group contained only 100 crashes and should
therefore be interpreted cautiously.

A four-class chi-square test showed a statistically significant association
between broad LCDB land-cover group and crash severity
(χ² = 675.68, df = 18, p < 0.001, Cramer's V = 0.0804). Because two expected
cells were below five and the minimum expected count was slightly below one,
this four-class inferential result is interpreted cautiously.

A supplementary Severe-versus-Non-Severe analysis also showed a statistically
significant association (χ² = 476.30, df = 6, p < 0.001, Cramer's V = 0.117).
All expected cell counts exceeded five in this analysis.

These findings indicate that land-cover environment contains measurable
information associated with recorded-crash severity. However, the association
does not imply causality because land cover may overlap with rurality,
speed environment, road design, and other crash-context characteristics.
The next experiment therefore tests whether LCDB provides incremental
predictive information after the existing CAS predictors are included.

In [9]:
# LCDB v6.0 Seed-42 Weighted-XGBoost Ablation

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Merge crash-level LCDB features into CAS

lcdb_features = lcdb_crash_final[
    [
        "cas_row_id",
        "LCDB_Name_clean",
        "LCDB_Group_clean",
        "Wetland_23",
        "LCDB_exact_match"
    ]
].copy()


model_data = cas.merge(
    lcdb_features,
    on="cas_row_id",
    how="left",
    validate="one_to_one"
)


print(
    "Modelling Dataset Validation"
)

print(
    "Rows:",
    len(model_data)
)

print(
    "Duplicate crash IDs:",
    model_data[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Missing LCDB grouped:",
    model_data[
        "LCDB_Group_clean"
    ]
    .isna()
    .sum()
)

print(
    "Missing LCDB detailed:",
    model_data[
        "LCDB_Name_clean"
    ]
    .isna()
    .sum()
)


# 2. Original 30-predictor CAS baseline

baseline_predictors = [
    "bicycle",
    "bus",
    "carStationWagon",
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "crashYear",
    "directionRoleDescription",
    "flatHill",
    "light",
    "moped",
    "motorcycle",
    "NumberOfLanes",
    "otherVehicleType",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "schoolBus",
    "speedLimit",
    "streetLight",
    "suv",
    "taxi",
    "trafficControl",
    "truck",
    "unknownVehicleType",
    "urban",
    "vanOrUtility",
    "weatherA",
    "weatherB"
]


baseline_categorical = [
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "directionRoleDescription",
    "flatHill",
    "light",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "streetLight",
    "trafficControl",
    "urban",
    "weatherA",
    "weatherB"
]


# 3. Define LCDB feature experiments

baseline_without_urban = [
    x
    for x in baseline_predictors
    if x != "urban"
]


baseline_cat_without_urban = [
    x
    for x in baseline_categorical
    if x != "urban"
]


experiments = {

    "CAS Only": {
        "predictors":
            baseline_predictors,

        "categorical":
            baseline_categorical
    },

    "CAS + LCDB Detailed": {
        "predictors":
            baseline_predictors
            + ["LCDB_Name_clean"],

        "categorical":
            baseline_categorical
            + ["LCDB_Name_clean"]
    },

    "CAS + LCDB Grouped": {
        "predictors":
            baseline_predictors
            + ["LCDB_Group_clean"],

        "categorical":
            baseline_categorical
            + ["LCDB_Group_clean"]
    },

    "CAS + LCDB Grouped + Wetland": {
        "predictors":
            baseline_predictors
            + [
                "LCDB_Group_clean",
                "Wetland_23"
            ],

        "categorical":
            baseline_categorical
            + [
                "LCDB_Group_clean",
                "Wetland_23"
            ]
    },

    "CAS Without Urban + LCDB Grouped": {
        "predictors":
            baseline_without_urban
            + ["LCDB_Group_clean"],

        "categorical":
            baseline_cat_without_urban
            + ["LCDB_Group_clean"]
    },

    "CAS Without Urban + LCDB Detailed": {
        "predictors":
            baseline_without_urban
            + ["LCDB_Name_clean"],

        "categorical":
            baseline_cat_without_urban
            + ["LCDB_Name_clean"]
    }
}


# 4. Encode four-class target

target = "crashSeverity"

label_encoder = LabelEncoder()

y_encoded = (
    label_encoder.fit_transform(
        model_data[
            target
        ]
    )
)


print(
    "\nTarget classes:"
)

for i, label in enumerate(
    label_encoder.classes_
):
    print(
        i,
        "=",
        label
    )


# 5. Use the same split for every experiment

SEED = 42

train_idx, test_idx = train_test_split(
    np.arange(
        len(model_data)
    ),
    test_size=0.20,
    random_state=SEED,
    stratify=y_encoded
)


y_train = y_encoded[
    train_idx
]

y_test = y_encoded[
    test_idx
]


sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)


# 6. Evaluation function

def evaluate_lcdb_experiment(
    experiment_name,
    predictors,
    categorical_cols
):

    numeric_cols = [
        col
        for col in predictors
        if col not in categorical_cols
    ]

    X = model_data[
        predictors
    ].copy()

    X_train = X.iloc[
        train_idx
    ]

    X_test = X.iloc[
        test_idx
    ]


    # Preprocessing

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )


    preprocess = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_cols
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_cols
            )
        ]
    )


    # Weighted XGBoost

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    )


    pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                preprocess
            ),
            (
                "model",
                model
            )
        ]
    )


    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=sample_weights
    )


    y_pred = pipeline.predict(
        X_test
    )


    class_recalls = recall_score(
        y_test,
        y_pred,
        labels=np.arange(
            len(
                label_encoder.classes_
            )
        ),
        average=None,
        zero_division=0
    )


    recall_lookup = dict(
        zip(
            label_encoder.classes_,
            class_recalls
        )
    )


    return {
        "Experiment":
            experiment_name,

        "Predictors":
            len(predictors),

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "Macro_F1":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Fatal_Recall":
            recall_lookup[
                "Fatal Crash"
            ],

        "Serious_Recall":
            recall_lookup[
                "Serious Crash"
            ]
    }


# 7. Run experiments

results = []

for experiment_name, config in experiments.items():

    print(
        "\nRunning:",
        experiment_name
    )

    result = evaluate_lcdb_experiment(
        experiment_name,
        config[
            "predictors"
        ],
        config[
            "categorical"
        ]
    )

    results.append(
        result
    )


# 8. Results table

lcdb_seed42_results = pd.DataFrame(
    results
)


baseline_macro = (
    lcdb_seed42_results.loc[
        lcdb_seed42_results[
            "Experiment"
        ] == "CAS Only",
        "Macro_F1"
    ]
    .iloc[0]
)


lcdb_seed42_results[
    "Macro_F1_Gain"
] = (
    lcdb_seed42_results[
        "Macro_F1"
    ]
    - baseline_macro
)


lcdb_seed42_results = (
    lcdb_seed42_results
    .sort_values(
        "Macro_F1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nLCDB Seed-42 Ablation Results"
)

display(
    lcdb_seed42_results.round(
        4
    )
)


# 9. Baseline reproduction check

baseline_row = (
    lcdb_seed42_results[
        lcdb_seed42_results[
            "Experiment"
        ] == "CAS Only"
    ]
    .iloc[0]
)


print(
    "\nBaseline Reproduction Check"
)

print(
    "Accuracy:",
    round(
        baseline_row[
            "Accuracy"
        ],
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        baseline_row[
            "Balanced_Accuracy"
        ],
        4
    )
)

print(
    "Macro F1:",
    round(
        baseline_row[
            "Macro_F1"
        ],
        4
    )
)

Modelling Dataset Validation
Rows: 34866
Duplicate crash IDs: 0
Missing LCDB grouped: 0
Missing LCDB detailed: 0

Target classes:
0 = Fatal Crash
1 = Minor Crash
2 = Non-Injury Crash
3 = Serious Crash

Running: CAS Only

Running: CAS + LCDB Detailed

Running: CAS + LCDB Grouped

Running: CAS + LCDB Grouped + Wetland

Running: CAS Without Urban + LCDB Grouped

Running: CAS Without Urban + LCDB Detailed

LCDB Seed-42 Ablation Results


,Experiment,Predictors,Accuracy,Balanced_Accuracy,Macro_F1,Fatal_Recall,Serious_Recall,Macro_F1_Gain
0,CAS + LCDB Detailed,31,0.4968,0.4144,0.3320,0.3385,0.3932,0.0048
1,CAS + LCDB Grouped,31,0.4958,0.4119,0.3314,0.3231,0.3972,0.0042
2,CAS Without Urban + LCDB Detailed,30,0.4940,0.4039,0.3287,0.3077,0.3792,0.0015
3,CAS Only,30,0.4900,0.4076,0.3272,0.3231,0.3892,0.0000
4,CAS Without Urban + LCDB Grouped,30,0.4913,0.4073,0.3271,0.3231,0.3912,-0.0002
5,CAS + LCDB Grouped + Wetland,32,0.4900,0.4044,0.3269,0.3077,0.3932,-0.0003



Baseline Reproduction Check
Accuracy: 0.49
Balanced Accuracy: 0.4076
Macro F1: 0.3272


In [10]:
# LCDB Five-Seed Robustness Experiment

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier

import numpy as np
import pandas as pd


# 1. Use the same five seeds as previous experiments

seeds = [
    42,
    7,
    21,
    84,
    123
]


# 2. Carry forward the two promising LCDB
# representations plus the CAS baseline

lcdb_robustness_sets = {

    "CAS Only": {
        "predictors":
            baseline_predictors,

        "categorical":
            baseline_categorical
    },

    "CAS + LCDB Detailed": {
        "predictors":
            baseline_predictors
            + [
                "LCDB_Name_clean"
            ],

        "categorical":
            baseline_categorical
            + [
                "LCDB_Name_clean"
            ]
    },

    "CAS + LCDB Grouped": {
        "predictors":
            baseline_predictors
            + [
                "LCDB_Group_clean"
            ],

        "categorical":
            baseline_categorical
            + [
                "LCDB_Group_clean"
            ]
    }
}


# 3. Function for one model at one seed

def run_lcdb_seed_model(
    model_name,
    predictors,
    categorical_cols,
    seed
):

    numeric_cols = [
        col
        for col in predictors
        if col not in categorical_cols
    ]


    X = model_data[
        predictors
    ].copy()


    # Same stratified split for all models
    # within this seed

    train_idx, test_idx = train_test_split(
        np.arange(
            len(model_data)
        ),
        test_size=0.20,
        random_state=seed,
        stratify=y_encoded
    )


    X_train = X.iloc[
        train_idx
    ]

    X_test = X.iloc[
        test_idx
    ]


    y_train = y_encoded[
        train_idx
    ]

    y_test = y_encoded[
        test_idx
    ]


    # Balanced sample weights

    sample_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )
    )


    # Preprocessing

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )


    preprocess = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_cols
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_cols
            )
        ]
    )


    # Weighted XGBoost

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=seed,
        n_jobs=-1
    )


    pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                preprocess
            ),
            (
                "model",
                model
            )
        ]
    )


    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=
            sample_weights
    )


    y_pred = pipeline.predict(
        X_test
    )


    # Class-specific recalls

    recalls = recall_score(
        y_test,
        y_pred,
        labels=np.arange(
            len(
                label_encoder.classes_
            )
        ),
        average=None,
        zero_division=0
    )


    recall_lookup = dict(
        zip(
            label_encoder.classes_,
            recalls
        )
    )


    return {

        "Seed":
            seed,

        "Experiment":
            model_name,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "Macro_F1":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Fatal_Recall":
            recall_lookup[
                "Fatal Crash"
            ],

        "Serious_Recall":
            recall_lookup[
                "Serious Crash"
            ]
    }


# 4. Run all five seeds

lcdb_seed_results = []


for seed in seeds:

    print(
        f"\nRunning seed: {seed}"
    )


    for model_name, config in (
        lcdb_robustness_sets.items()
    ):

        print(
            "Running:",
            model_name
        )


        result = run_lcdb_seed_model(
            model_name=
                model_name,

            predictors=
                config[
                    "predictors"
                ],

            categorical_cols=
                config[
                    "categorical"
                ],

            seed=
                seed
        )


        lcdb_seed_results.append(
            result
        )


lcdb_robustness_df = pd.DataFrame(
    lcdb_seed_results
)


# 5. Per-seed results

print(
    "\nLCDB Per-Seed Results"
)

display(
    lcdb_robustness_df.round(4)
)


# 6. Mean and standard deviation

metrics = [
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Fatal_Recall",
    "Serious_Recall"
]


summary_rows = []


for model_name in (
    lcdb_robustness_sets.keys()
):

    subset = (
        lcdb_robustness_df[
            lcdb_robustness_df[
                "Experiment"
            ] == model_name
        ]
    )


    row = {
        "Experiment":
            model_name
    }


    for metric in metrics:

        row[
            metric + "_Mean"
        ] = subset[
            metric
        ].mean()


        row[
            metric + "_SD"
        ] = subset[
            metric
        ].std()


    summary_rows.append(
        row
    )


lcdb_robustness_summary = (
    pd.DataFrame(
        summary_rows
    )
)


# 7. Mean Macro F1 gain against CAS baseline

baseline_mean_macro = (
    lcdb_robustness_summary.loc[
        lcdb_robustness_summary[
            "Experiment"
        ] == "CAS Only",
        "Macro_F1_Mean"
    ]
    .iloc[0]
)


lcdb_robustness_summary[
    "Macro_F1_Mean_Gain"
] = (
    lcdb_robustness_summary[
        "Macro_F1_Mean"
    ]
    - baseline_mean_macro
)


lcdb_robustness_summary = (
    lcdb_robustness_summary
    .sort_values(
        "Macro_F1_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nLCDB Five-Seed Robustness Summary"
)

display(
    lcdb_robustness_summary.round(4)
)


# 8. How many seeds improved Macro F1?

macro_pivot = (
    lcdb_robustness_df
    .pivot(
        index="Seed",
        columns="Experiment",
        values="Macro_F1"
    )
)


print(
    "\nMacro F1 Improvement by Seed"
)


for model_name in [
    "CAS + LCDB Detailed",
    "CAS + LCDB Grouped"
]:

    gains = (
        macro_pivot[
            model_name
        ]
        -
        macro_pivot[
            "CAS Only"
        ]
    )


    print(
        "\n",
        model_name
    )


    print(
        gains.round(4)
    )


    print(
        "Improved seeds:",
        int(
            (
                gains > 0
            ).sum()
        ),
        "/",
        len(seeds)
    )


    print(
        "Mean Macro F1 gain:",
        round(
            gains.mean(),
            4
        )
    )


# 9. Previous CAS robustness reproduction check

baseline_summary = (
    lcdb_robustness_summary[
        lcdb_robustness_summary[
            "Experiment"
        ] == "CAS Only"
    ]
    .iloc[0]
)


print(
    "\nFive-Seed CAS Baseline Check"
)

print(
    "Accuracy mean:",
    round(
        baseline_summary[
            "Accuracy_Mean"
        ],
        4
    )
)

print(
    "Balanced Accuracy mean:",
    round(
        baseline_summary[
            "Balanced_Accuracy_Mean"
        ],
        4
    )
)

print(
    "Macro F1 mean:",
    round(
        baseline_summary[
            "Macro_F1_Mean"
        ],
        4
    )
)

print(
    "Fatal recall mean:",
    round(
        baseline_summary[
            "Fatal_Recall_Mean"
        ],
        4
    )
)

print(
    "Serious recall mean:",
    round(
        baseline_summary[
            "Serious_Recall_Mean"
        ],
        4
    )
)


Running seed: 42
Running: CAS Only
Running: CAS + LCDB Detailed
Running: CAS + LCDB Grouped

Running seed: 7
Running: CAS Only
Running: CAS + LCDB Detailed
Running: CAS + LCDB Grouped

Running seed: 21
Running: CAS Only
Running: CAS + LCDB Detailed
Running: CAS + LCDB Grouped

Running seed: 84
Running: CAS Only
Running: CAS + LCDB Detailed
Running: CAS + LCDB Grouped

Running seed: 123
Running: CAS Only
Running: CAS + LCDB Detailed
Running: CAS + LCDB Grouped

LCDB Per-Seed Results


,Seed,Experiment,Accuracy,Balanced_Accuracy,Macro_F1,Fatal_Recall,Serious_Recall
0,42,CAS Only,0.4900,0.4076,0.3272,0.3231,0.3892
1,42,CAS + LCDB Detailed,0.4968,0.4144,0.3320,0.3385,0.3932
2,42,CAS + LCDB Grouped,0.4958,0.4119,0.3314,0.3231,0.3972
3,7,CAS Only,0.4920,0.4111,0.3284,0.3385,0.3892
4,7,CAS + LCDB Detailed,0.4961,0.3808,0.3243,0.2308,0.3613
5,7,CAS + LCDB Grouped,0.4963,0.3973,0.3266,0.2923,0.3673
6,21,CAS Only,0.4991,0.4217,0.3351,0.3538,0.4012
7,21,CAS + LCDB Detailed,0.5072,0.4159,0.3377,0.3231,0.3912
8,21,CAS + LCDB Grouped,0.5052,0.4160,0.3364,0.3231,0.3972
9,84,CAS Only,0.4971,0.4024,0.3315,0.2615,0.4172



LCDB Five-Seed Robustness Summary


,Experiment,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD,Fatal_Recall_Mean,Fatal_Recall_SD,Serious_Recall_Mean,Serious_Recall_SD,Macro_F1_Mean_Gain
0,CAS + LCDB Grouped,0.5001,0.0040,0.4066,0.0125,0.3314,0.0047,0.3015,0.0337,0.3912,0.0185,0.0025
1,CAS + LCDB Detailed,0.5001,0.0047,0.4036,0.0158,0.3309,0.0050,0.2954,0.0467,0.3852,0.0171,0.0021
2,CAS Only,0.4940,0.0039,0.4062,0.0123,0.3288,0.0049,0.3077,0.0435,0.3944,0.0157,0.0000



Macro F1 Improvement by Seed

 CAS + LCDB Detailed
Seed
7     -0.0042
21     0.0026
42     0.0048
84     0.0008
123    0.0064
dtype: float64
Improved seeds: 4 / 5
Mean Macro F1 gain: 0.0021

 CAS + LCDB Grouped
Seed
7     -0.0019
21     0.0013
42     0.0042
84     0.0042
123    0.0048
dtype: float64
Improved seeds: 4 / 5
Mean Macro F1 gain: 0.0025

Five-Seed CAS Baseline Check
Accuracy mean: 0.494
Balanced Accuracy mean: 0.4062
Macro F1 mean: 0.3288
Fatal recall mean: 0.3077
Serious recall mean: 0.3944


### LCDB Robustness and Feature-Representation Finding

The five-seed evaluation showed that the broader LCDB land-cover groups gave the most reliable land-cover representation for the model.

The CAS-only weighted XGBoost model achieved a mean Macro F1 of 0.3288. When grouped LCDB land cover was added, the mean Macro F1 increased to 0.3314. It also improved Macro F1 in four out of the five random-seed runs.

Mean accuracy increased from 0.4940 to 0.5001. Balanced Accuracy stayed almost the same, with a small increase from 0.4062 to 0.4066.

However, the improvement was not seen in every metric. Mean Fatal-crash recall decreased slightly from 0.3077 to 0.3015, while Serious-crash recall decreased from 0.3944 to 0.3912. So, although grouped LCDB helped overall model performance slightly, it did not improve all crash-severity classes.

The detailed 34-class LCDB version achieved a mean Macro F1 of 0.3309, compared with 0.3314 for the grouped version. Because the grouped representation performed slightly better overall, was easier to interpret, and avoided having many low-frequency land-cover classes, it was selected for the later combined-feature experiments.

Overall, LCDB seems to provide some useful additional environmental context beyond the existing CAS predictors. The improvement is small but fairly consistent across the five seeds. These results only show an association and some predictive contribution, and they do not mean that land-cover type directly causes differences in crash severity.

In [11]:
# Save reusable LCDB v6.0 crash enrichment

LCDB_ENRICHED_PATH = (
    PROJECT_ROOT
    / "data"
    / "lcdb"
    / "CAS_LCDB_v6_enriched.csv"
)


lcdb_export = lcdb_crash_final[
    [
        "cas_row_id",
        "crashSeverity",
        "Name_2023",
        "Class_2023",
        "LCDB_Name_clean",
        "LCDB_Group_clean",
        "Wetland_23",
        "Onshore_23",
        "LCDB_UID",
        "LCDB_exact_match"
    ]
].copy()


lcdb_export.to_csv(
    LCDB_ENRICHED_PATH,
    index=False
)


print(
    "Saved:"
)

print(
    LCDB_ENRICHED_PATH
)


print(
    "\nShape:"
)

print(
    lcdb_export.shape
)


print(
    "\nDuplicate cas_row_id:"
)

print(
    lcdb_export[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


print(
    "\nMissing selected LCDB group:"
)

print(
    lcdb_export[
        "LCDB_Group_clean"
    ]
    .isna()
    .sum()
)


print(
    "\nExact LCDB matches:"
)

print(
    lcdb_export[
        "LCDB_exact_match"
    ]
    .value_counts()
    .sort_index()
)

Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\lcdb\CAS_LCDB_v6_enriched.csv

Shape:
(34866, 10)

Duplicate cas_row_id:
0

Missing selected LCDB group:
0

Exact LCDB matches:
LCDB_exact_match
0       63
1    34803
Name: count, dtype: int64
